# STAGE C/5 — **v3**: Head A (NER) + Head B (ASSERTION HỌC)

**Add Input (BẮT BUỘC):** **Stage A** (dapt) + **Stage B** (silver, llm) + **Stage D** (kb).
**Settings:** GPU **T4 (1 con)** + Internet ON. **~50′**. Xong → **Commit** cho Stage E.

v3: **Head A** = NER trên consensus (distant KB abbrev-aware + silver khử-rác + template);
**Head B** = assertion học từ silver (ViHealthBERT) — thay ConText regex.


In [ ]:
# Clone repo
import os, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # 1 GPU: tránh DataParallel chậm
REPO = "https://github.com/Khanhhh239/Medical_Retrieve.git"
D = "/kaggle/working/Medical_Retrieve"
if not os.path.isdir(D):
    subprocess.run(["git","clone","-q",REPO,D], check=True)
else:
    subprocess.run(["git","-C",D,"pull","-q"])
os.chdir(D); print('cwd:', os.getcwd())


In [ ]:
!pip install -q rapidfuzz pyyaml regex accelerate sentencepiece


In [ ]:
# Tìm artifact từ stage trước (đã Add Input)
import glob, os, shutil
def find_input(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None
def find_model(name):
    # model dir = thư mục có config.json (TRÁNH khớp nhầm src/ner là code)
    hits = [h for h in glob.glob(f"/kaggle/input/**/{name}/config.json", recursive=True)
            if "/src/" not in h]
    return os.path.dirname(hits[0]) if hits else None


In [ ]:
import glob, os, shutil
DAPT = find_model('dapt') or 'xlm-roberta-large'
SILVER = find_input('silver.jsonl') or ''
LLM = find_input('llm.jsonl') or ''
KB = find_input('kb')
if KB:                                   # KB (Stage D) cho distant + linking
    for f in glob.glob(KB + '/*'):
        if os.path.isfile(f): shutil.copy(f, 'data/kb/' + os.path.basename(f))
    print('KB loaded:', os.listdir('data/kb'))
print('DAPT:', DAPT, '| SILVER:', bool(SILVER), '| LLM:', bool(LLM), '| KB:', bool(KB))
if DAPT == 'xlm-roberta-large': print('!! chua Add Input Stage A')
if not KB: print('!! chua Add Input Stage D (kb)')


In [ ]:
# v3: distant (KB abbrev-aware) + khử-rác silver/llm
_a = (f' --silver {SILVER}' if SILVER else '') + (f' --llm {LLM}' if LLM else '')
!python scripts/prep_v2.py{_a} --out_dir data/v2


In [ ]:
# template (sàn)
!python scripts/gen_synth.py --n_train 4000 --n_dev 400


In [ ]:
# HEAD A — NER trên consensus [distant + silver_clean + llm_clean + template]
cands = ['data/v2/distant.jsonl','data/v2/silver_clean.jsonl','data/v2/llm_clean.jsonl','data/synthetic/train.jsonl']
train_arg = ','.join([p for p in cands if os.path.exists(p)])
print('TRAIN A:', train_arg)
!python scripts/train_ner.py --model {DAPT} --train {train_arg} \
   --epochs 3 --batch_size 8 --grad_accum 2 --max_length 384 --optim adafactor \
   --out models/ner


In [ ]:
# HEAD B — ASSERTION học từ silver (ViHealthBERT). Lỗi tích hợp -> đổi --model xlm-roberta-base
_s = ','.join([p for p in [SILVER, LLM] if p])
!python scripts/train_assert.py --train {_s} --out models/assert \
   || echo 'assert loi -> chay lai: !python scripts/train_assert.py --model xlm-roberta-base --train {_s} --out models/assert'


In [ ]:
# Lưu 2 model ra /kaggle/working để Commit
import shutil, os
for name in ['ner','assert']:
    src, dst = 'models/'+name, '/kaggle/working/'+name
    if os.path.isdir(src):
        shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst); print('SAVED', dst)
    else: print('!! thiếu', src)


**OOM?** `--batch_size 4 --grad_accum 4`. **Head B lỗi ViHealthBERT** → đổi `--model xlm-roberta-base`. prep_v2 cần KB (Stage D).
